# Bimodularity Community Detection on a FlyWire Slice

This notebook applies the **bimodularity** technique from Cionca et al. (directed community detection) to a **small slice** of the FlyWire connectome.

It:
- selects a 100–200 node slice with a “good” community structure (via Louvain on the **symmetrized** graph),
- runs **bimodularity** on the **directed** slice (sender vs receiver communities),
- compares to **conventional modularity** (Louvain modularity on the symmetrized graph).

Implementation lives in `notebooks/bimodularity_flywire.py` and is imported here.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "notebooks").exists():
    repo_root = cwd
elif cwd.name == "notebooks" and (cwd.parent / "data").exists():
    repo_root = cwd.parent
else:
    repo_root = cwd
sys.path.insert(0, str(repo_root / "notebooks"))

import bimodularity_flywire as bi

sns.set_context("talk")
sns.set_style("white")
print("repo_root:", repo_root)

## Load the precomputed FlyWire directed subgraph

Uses `data/preprocessed/subgraph.pkl` (generated in `notebooks/network_analysis.ipynb`).

In [ ]:
graph_path = repo_root / "data/preprocessed/subgraph.pkl"
G_full = bi.load_flywire_subgraph_pickle(graph_path)
print(type(G_full), "nodes=", G_full.number_of_nodes(), "edges=", G_full.number_of_edges(), "directed=", G_full.is_directed())

## Pick a 100–200 node slice with “good” communities

Heuristic:
- run Louvain on the **symmetrized** graph,
- among communities sized in `[nodes_min, nodes_max]`, pick the one with high directed internal density and low outgoing conductance.

In [ ]:
nodes_min, nodes_max = 120, 180
seed = 1

slice_info = bi.pick_slice_via_louvain(G_full, target_min=nodes_min, target_max=nodes_max, seed=seed, weight="weight")
print("Selected slice:", f"n={len(slice_info.nodes)}", "|", slice_info.reason)

G = G_full.subgraph(slice_info.nodes).copy()
nodes = list(G.nodes())
A = bi.adjacency_from_digraph(G, nodes, weight="weight")
print("Slice edges:", G.number_of_edges(), "total weight:", float(A.sum()))

## Baseline: conventional modularity (Louvain on symmetrized graph)

We compute:
- `Q_und`: modularity of Louvain partition on symmetrized graph,
- `Q_dir`: directed modularity score of the same partition under the out–in null model,
- within-community directed weight fraction.

In [ ]:
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity

gamma = 1.0
und = bi.symmetrize_to_undirected(G, weight="weight")
mod_comms = louvain_communities(und, weight="weight", seed=seed)
Q_und = modularity(und, mod_comms, weight="weight")
mod_labels = bi._labels_from_communities(nodes, mod_comms)
Q_dir = bi.directed_modularity_score(A, mod_labels, gamma=gamma)
within_frac_mod = float(A[mod_labels[:, None] == mod_labels[None, :]].sum() / max(1.0, A.sum()))

print(f"Q_und={Q_und:.4f}  Q_dir={Q_dir:.4f}  within_frac={within_frac_mod:.3f}  n_comms={len(mod_comms)}")

## Bimodularity (directed sender vs receiver communities)

We compute the asymmetric modularity matrix (out–in null model):

$$B_{ij} = A_{ij} - \gamma \frac{k^{out}_i k^{in}_j}{m}$$

Then (following the spirit of the paper’s SVD-based approach):
- take SVD of `B`,
- embed edges `(i→j)` using top singular vectors,
- K-means cluster edges,
- assign each node a sender-cluster and receiver-cluster by strongest participation.

We try a small range of `K` and pick the best by `Q_bi`.

In [ ]:
svd_rank = 6
min_edge_weight = 5.0
max_edges = 8000
K_values = list(range(3, 9))

results = []
for K in K_values:
    res = bi.bimodularity_partition_via_edge_kmeans(
        A,
        svd_rank=svd_rank,
        n_clusters=K,
        min_edge_weight=min_edge_weight,
        max_edges=max_edges,
        seed=seed,
        gamma=gamma,
    )
    within_frac_bi = float(A[res.send_label[:, None] == res.recv_label[None, :]].sum() / max(1.0, A.sum()))
    results.append({"K": K, "Q_bi": res.q_bimod, "within_frac": within_frac_bi, "role_mismatch": float(np.mean(res.send_label != res.recv_label))})

df = pd.DataFrame(results).sort_values("Q_bi", ascending=False)
df

In [ ]:
best_K = int(df.iloc[0]["K"])
best = bi.bimodularity_partition_via_edge_kmeans(
    A,
    svd_rank=svd_rank,
    n_clusters=best_K,
    min_edge_weight=min_edge_weight,
    max_edges=max_edges,
    seed=seed,
    gamma=gamma,
)
within_frac_bi = float(A[best.send_label[:, None] == best.recv_label[None, :]].sum() / max(1.0, A.sum()))
print(f"Best K={best_K}  Q_bi={best.q_bimod:.4f}  within_frac={within_frac_bi:.3f}  gain={within_frac_bi - within_frac_mod:+.3f}")

## Visual comparison

- Left: reorder adjacency by modularity communities (symmetrized Louvain)
- Right: reorder adjacency by bimodularity sender/receiver roles

In [ ]:
order_mod = bi._order_by_label_then_strength(A, mod_labels)
A_mod = A[np.ix_(order_mod, order_mod)]

row_order_bi = bi._order_by_label_then_strength(A, best.send_label)
col_order_bi = bi._order_by_label_then_strength(A.T, best.recv_label)
A_bi = A[np.ix_(row_order_bi, col_order_bi)]

vmax = np.percentile(A[A > 0], 99) if np.any(A > 0) else 1.0
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
sns.heatmap(np.log1p(A_mod), ax=axes[0], cmap="mako", cbar=False, vmin=0, vmax=np.log1p(vmax))
axes[0].set_title("Conventional modularity (sym Louvain)\nlog(1 + A) reordered")
axes[0].set_xlabel("nodes")
axes[0].set_ylabel("nodes")

sns.heatmap(np.log1p(A_bi), ax=axes[1], cmap="mako", cbar=False, vmin=0, vmax=np.log1p(vmax))
axes[1].set_title("Bimodularity (send vs recv)\nlog(1 + A) reordered")
axes[1].set_xlabel("receivers")
axes[1].set_ylabel("senders")
plt.show()

## Directed block matrices

Summarize directed weight between groups:
- modularity communities (left)
- bimodularity sender→receiver clusters (right)

In [ ]:
block_mod = bi._coarsened_weight_matrix(A, mod_labels, mod_labels)
block_bi = bi._coarsened_weight_matrix(A, best.send_label, best.recv_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.heatmap(block_mod, ax=axes[0], cmap="rocket_r", square=True)
axes[0].set_title("Directed weight between Louvain communities")
axes[0].set_xlabel("to community")
axes[0].set_ylabel("from community")

sns.heatmap(block_bi, ax=axes[1], cmap="rocket_r", square=True)
axes[1].set_title("Directed weight between bimodularity roles\n(send cluster → recv cluster)")
axes[1].set_xlabel("to (recv cluster)")
axes[1].set_ylabel("from (send cluster)")
plt.show()

## Optional: attach metadata (cell types / labels)

Uses `data/preprocessed/processed_nodes.csv` if present.

In [ ]:
nodes_df_path = repo_root / "data/preprocessed/processed_nodes.csv"
if nodes_df_path.exists():
    nodes_df = pd.read_csv(nodes_df_path)
    meta = nodes_df.set_index("id").reindex(nodes)
    meta = meta.assign(send_cluster=best.send_label, recv_cluster=best.recv_label, mod_cluster=mod_labels)
    display(meta.head())
    if "community_label" in meta.columns:
        display(meta["community_label"].fillna("NA").value_counts().head(15))
else:
    print("No metadata file:", nodes_df_path)